In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 51.9 MB/s eta 0:00:00


In [ ]:
import os
from pathlib import Path

DATASET_ROOT = '/content/drive/MyDrive/PaperMdDataset'
JSON_DIR     = f'{DATASET_ROOT}/labels/json'
LABELS_DIR   = f'{DATASET_ROOT}/labels/train'
IMAGES_DIR   = f'{DATASET_ROOT}/images/train'
DATA_YAML    = f'{DATASET_ROOT}/data.yaml'
PROJECT_DIR  = '/content/runs/detect'
RUN_NAME     = 'papermd_train'
MODELS_DIR   = f'{DATASET_ROOT}/models'

CLASS_MAP = {
    'Abandon':    0,
    'Plain_Text': 1,
    'Formula':    2,
    'Figure':     3,
    'Table':      4,
    'Link':       5,
    'Callout':    6,
}


In [ ]:
# ── Label conversion ──────────────────────────────────────────────────────────

import json

if os.path.exists(LABELS_DIR):
    !rm -rf {LABELS_DIR}/*.txt
os.makedirs(LABELS_DIR, exist_ok=True)

files = [f for f in os.listdir(JSON_DIR) if os.path.isfile(os.path.join(JSON_DIR, f))]
print(f"Found {len(files)} annotation files. Starting conversion...\n")

converted, skipped, errors = 0, 0, 0

for file_name in files:
    file_path = os.path.join(JSON_DIR, file_name)
    try:
        with open(file_path, 'r') as f:
            data = json.load(f)

        image_path = data.get('task', {}).get('data', {}).get('image', '')
        if not image_path:
            skipped += 1
            continue

        label_name = Path(image_path).stem + '.txt'
        results    = data.get('result', [])
        yolo_lines = []

        for res in results:
            val        = res.get('value', {})
            label_list = val.get('rectanglelabels', [])
            if not label_list:
                continue

            label = label_list[0]
            if label not in CLASS_MAP:
                print(f"  ⚠ Unknown label '{label}' in {file_name} — skipping annotation")
                continue

            class_id = CLASS_MAP[label]

            # Label Studio % → YOLO normalised 0-1
            w        = val['width']  / 100
            h        = val['height'] / 100
            x_center = (val['x'] / 100) + (w / 2)
            y_center = (val['y'] / 100) + (h / 2)

            # Clamp to valid range
            x_center = max(0.0, min(1.0, x_center))
            y_center = max(0.0, min(1.0, y_center))
            w        = max(0.0, min(1.0, w))
            h        = max(0.0, min(1.0, h))

            yolo_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}")

        if yolo_lines:
            with open(os.path.join(LABELS_DIR, label_name), 'w') as out_f:
                out_f.write("\n".join(yolo_lines))
            converted += 1
        else:
            skipped += 1

    except Exception as e:
        errors += 1
        print(f"  ✗ Error processing {file_name}: {e}")

print(f"\nFinished. Converted: {converted} | Skipped: {skipped} | Errors: {errors}")


In [ ]:
# ── Dataset validation ────────────────────────────────────────────────────────

from collections import Counter

image_stems = {f.stem for f in Path(IMAGES_DIR).glob('*.jpg')}
label_stems = {f.stem for f in Path(LABELS_DIR).glob('*.txt')}

matched   = image_stems & label_stems
imgs_only = image_stems - label_stems
lbls_only = label_stems - image_stems

print(f"Matched pairs : {len(matched)}")
if imgs_only:
    print(f"Images without labels : {len(imgs_only)}")
if lbls_only:
    print(f"Labels without images : {len(lbls_only)}")

class_names = {v: k for k, v in CLASS_MAP.items()}
counts      = Counter()

for txt in Path(LABELS_DIR).glob('*.txt'):
    for line in txt.read_text().splitlines():
        parts = line.strip().split()
        if not parts:
            continue
        if parts[0].isdigit():
            counts[int(parts[0])] += 1
        elif parts[0] in CLASS_MAP:
            counts[CLASS_MAP[parts[0]]] += 1

print(f"\nClass distribution:")
for cid in sorted(class_names):
    bar = '█' * (counts[cid] // 2)
    print(f"  {class_names[cid]:<20} {counts[cid]:>4}  {bar}")

In [ ]:
# ── Training ──────────────────────────────────────────────────────────────────

from ultralytics import YOLO

if not os.path.exists(DATA_YAML):
    print(f"Error: {DATA_YAML} not found.")
else:
    model = YOLO('yolo11n.pt')
    results = model.train(
        data=DATA_YAML,
        epochs=100,
        imgsz=1280,
        batch=8,
        augment=True,
        project=PROJECT_DIR,
        name=RUN_NAME,
        exist_ok=True,
        patience=20,
    )
    print(f"\nBest mAP50: {results.results_dict.get('metrics/mAP50(B)', 'N/A'):.4f}")


In [ ]:
# ── Save .pt weights to Drive ─────────────────────────────────────────────────

import shutil
from datetime import datetime

os.makedirs(MODELS_DIR, exist_ok=True)

pt_source = os.path.join(PROJECT_DIR, RUN_NAME, 'weights/best.pt')

if not os.path.exists(pt_source):
    print(f"Error: weights not found at {pt_source}")
else:
    timestamp   = datetime.now().strftime("%m%d_%H%M")
    img_count   = len(list(Path(LABELS_DIR).glob('*.txt')))
    pt_filename = f"{timestamp}_n{img_count}.pt"
    pt_dest     = os.path.join(MODELS_DIR, pt_filename)

    shutil.copy(pt_source, pt_dest)
    print(f"✓ .pt  saved → {pt_filename}")


In [ ]:
# ── Export to ONNX ────────────────────────────────────────────────────────────

    best_model = YOLO(pt_source)

    best_model.export(
        format='onnx',
        imgsz=1280,
        simplify=True,
        opset=12,
    )

    onnx_source   = pt_source.replace('.pt', '.onnx')
    onnx_filename = pt_filename.replace('.pt', '.onnx')
    onnx_dest     = os.path.join(MODELS_DIR, onnx_filename)

    if os.path.exists(onnx_source):
        shutil.copy(onnx_source, onnx_dest)
        print(f"✓ .onnx saved → {onnx_filename}")
    else:
        print(f"Error: ONNX export file not found at {onnx_source}")

    print(f"\nModel release: {timestamp}_n{img_count}")
    print(f"  Classes : {len(CLASS_MAP)} ({', '.join(CLASS_MAP.keys())})")
    print(f"  Images  : {img_count}")